# Splitting Data into Training and Testing Sets

**Before a model can be trusted, it must be evaluated on data it has never seen before. If you train and evaluate on the same dataset, you are not measuring learning — you are measuring memorization.**

## Overview

Splitting data into training and testing sets is the foundational step that ensures honest evaluation. It creates a boundary between the data used to learn patterns and the data used to measure generalization.

**Without this separation, performance metrics are misleading and model quality cannot be assessed reliably.**

### By the end of this lesson, you will:

- Understand why data splitting is essential for valid evaluation
- Implement correct train-test splits using scikit-learn
- Apply stratified splitting for classification problems
- Recognize and prevent data leakage during splitting
- Use time-based splitting for temporal data
- Validate splits and detect imbalance
- Understand the role of cross-validation
- Document splitting strategies for reproducibility

## Why Data Splitting Is Necessary

Machine learning models are designed to **generalize** — to perform well on unseen data. If you evaluate on the same data used for training:

- The model may **memorize patterns** rather than learn generalizable rules
- **Overfitting goes undetected** — the model learns noise and data artifacts
- **Metrics appear artificially high** — performance on training data is always optimistic
- **Deployment performance collapses** — the model fails on real-world data

### The Real-World Analogy

A proper train-test split simulates the real-world scenario:

- **Training data** represents historical data (what has already happened)
- **Testing data** represents future or unseen examples (new situations)
- **The test set acts as a proxy for production** (how the model will be used)

### What Valid Evaluation Tells You

- **Strong test performance** → evidence of generalization
- **Large gap between training and test performance** → likely overfitting
- **Similar performance on both** → model is learning patterns, not memorizing

**Without proper splitting, you have no reliable measure of how the model will perform in production.**

## Training Set vs Testing Set

### Training Set

The training set is used to:

- Fit preprocessing transformations (compute mean, std, etc.)
- Learn model parameters (weights, splits, etc.)
- Tune hyperparameters (via cross-validation)

**The model is allowed to see this data. It learns from it.**

### Testing Set

The testing set is used to:

- Evaluate final model performance
- Compute unbiased metrics
- Simulate unseen real-world inputs

**The model must NEVER learn from this data. It must be held out completely.**

### The Sacred Rule

**The test set must remain untouched until final evaluation.**

This means:
- Never use test data for hyperparameter tuning
- Never use test statistics for preprocessing
- Never make decisions based on test set performance during development
- Only perform final evaluation once, at the very end

## Standard Train-Test Split

### Common Split Ratios

The most common split proportions are:

- **70% training / 30% testing** — smaller dataset or more conservative
- **80% training / 20% testing** — standard, widely used
- **90% training / 10% testing** — large dataset or when test size can be smaller

The choice depends on dataset size, complexity, and stability needs.

### Implementation with scikit-learn

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42     # ensures reproducibility
)
```

### Important Parameters

**test_size** (float or int)
- Proportion or number of samples for testing
- `test_size=0.2` → 20% testing
- `test_size=500` → exactly 500 samples for testing

**random_state** (int)
- Seed for random number generator
- Ensures that the **same split can be reproduced later**
- Essential for reproducibility and debugging
- Any fixed value works; 42 is conventional

**shuffle** (bool, default=True)
- Whether to shuffle before splitting
- Should be False for time-series data

### Why random_state Matters

Without a fixed random_state, each run produces a different split:

```python
# Run 1: Different split
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Run 2: Different split again
X_train, X_test, y_train, y_test = train_test_split(X, y)

# With fixed random_state, reproducible:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
# Same split every time
```

**Always use a fixed random_state for reproducibility.**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create sample dataset
np.random.seed(42)
n_samples = 1000

X = pd.DataFrame({
    'feature_1': np.random.randn(n_samples),
    'feature_2': np.random.randn(n_samples) * 2 + 1,
    'feature_3': np.random.randint(0, 10, n_samples)
})

y = pd.Series(
    np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3]),
    name='target'
)

print("Full dataset shape:", X.shape)
print("Target distribution (full):")
print(y.value_counts(normalize=True))
print()

# Standard train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("=" * 60)
print("AFTER TRAIN-TEST SPLIT (80-20)")
print("=" * 60)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print()
print("Target distribution (training):")
print(y_train.value_counts(normalize=True))
print()
print("Target distribution (testing):")
print(y_test.value_counts(normalize=True))

## Stratified Splitting (For Classification)

### The Problem with Random Splitting

In classification problems, especially imbalanced ones, simple random splitting can distort class distribution.

**Example:**

If 10% of samples are positive (class 1) and 90% are negative (class 0), random splitting might produce:

- Training set: 12% positive, 88% negative
- Testing set: 5% positive, 95% negative

**Result:** Evaluation instability. The test set has a different class balance than training, making performance metrics harder to interpret.

### Stratified Splitting Solution

Stratified splitting preserves class proportions in both training and testing sets:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # This ensures class balance preservation
)
```

### When to Use Stratification

- **Binary classification** → Always use stratification
- **Multi-class classification** → Always use stratification
- **Regression** → Not applicable (continuous target)
- **Imbalanced classification** → Essential
- **Balanced classification** → Still recommended for consistency

**Best practice:** Use stratification for all classification problems.

In [ ]:
# Demonstrate stratified vs random splitting

print("=" * 60)
print("STRATIFIED SPLITTING COMPARISON")
print("=" * 60)
print()

# Random split (without stratification)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=None  # No stratification
)

# Stratified split
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Stratified
)

print("Full dataset class distribution:")
print(y.value_counts(normalize=True))
print()

print("Random split (without stratification):")
print("  Training:", y_train_rand.value_counts(normalize=True).to_dict())
print("  Testing:", y_test_rand.value_counts(normalize=True).to_dict())
print()

print("Stratified split:")
print("  Training:", y_train_strat.value_counts(normalize=True).to_dict())
print("  Testing:", y_test_strat.value_counts(normalize=True).to_dict())
print()

print("Key insight: Stratified split preserves class balance!")

## The Critical Rule: Split Before Fitting

### The Most Common Mistake

**The single most important rule in data splitting:**

**Split the data BEFORE fitting any preprocessing transformations.**

### Incorrect Approach (Data Leakage)

```python
# WRONG - This causes data leakage
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fitted on ALL data

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**Why this fails:**
- The scaler was fitted on the entire dataset, including test data
- The test set statistics (mean, std) influenced the scaler parameters
- The model indirectly learns from test data through scaling
- Test set mean and std are "leaked" into training data
- **Evaluation becomes invalid** — test set is no longer truly unseen

### Correct Approach

```python
# RIGHT - No data leakage
from sklearn.preprocessing import StandardScaler

# Step 1: Split first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Step 3: Apply scaler to test data (no fitting)
X_test_scaled = scaler.transform(X_test)
```

**Why this works:**
- Scaler learns only from training data
- Test data is transformed using training-based parameters
- No test information leaks into preprocessing
- Test set remains truly unseen
- **Evaluation is valid and unbiased**

### General Pattern

For ANY preprocessing that requires fitting:

1. **Split first**
2. **Fit on training data** (`fit()` or `fit_transform()`)
3. **Apply to test data** (`transform()` only, never `fit_transform()`)

This applies to:
- Scaling (StandardScaler, MinMaxScaler)
- Encoding (OneHotEncoder, LabelEncoder)
- Imputation (SimpleImputer)
- Any fitting operation

In [ ]:
# Demonstrate the leakage problem

print("=" * 60)
print("DEMONSTRATING DATA LEAKAGE")
print("=" * 60)
print()

# Incorrect approach: fit before split
print("INCORRECT: Scale before splitting")
print("-" * 60)
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)  # Fitted on entire dataset!

X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X_scaled_wrong, y, test_size=0.2, random_state=42
)

print(f"Scaler mean (fitted on all data): {scaler_wrong.mean_}")
print(f"Training set mean: {X_train_wrong.mean(axis=0)}")
print(f"Testing set mean: {X_test_wrong.mean(axis=0)}")
print("❌ Test set statistics influenced the scaler!")
print()

# Correct approach: split before fit
print("CORRECT: Split before scaling")
print("-" * 60)
X_train_correct, X_test_correct, y_train_correct, y_test_correct = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_correct = StandardScaler()
X_train_correct_scaled = scaler_correct.fit_transform(X_train_correct)  # Fitted on training only
X_test_correct_scaled = scaler_correct.transform(X_test_correct)  # Applied to test

print(f"Scaler mean (fitted on training only): {scaler_correct.mean_}")
print(f"Training set mean after scaling: {X_train_correct_scaled.mean(axis=0)}")
print(f"Testing set mean after scaling: {X_test_correct_scaled.mean(axis=0)}")
print("✓ Test set was not used to fit the scaler!")
print()

print("Key difference:")
print(f"  Incorrect approach test mean shape: {X_test_wrong.mean(axis=0)}")
print(f"  Correct approach test mean shape: {X_test_correct_scaled.mean(axis=0)}")

## Time-Based Splitting (For Temporal Data)

### Why Random Shuffling Fails for Time-Series

For time-series or chronological datasets, **random shuffling is incorrect and creates unrealistic evaluation.**

**Example: Predicting future sales based on historical data**

If you randomly shuffle:
- Training data contains future information
- The model sees the future while learning the past
- Test set includes historical data
- Evaluation is impossible to interpret

### Correct Approach: Chronological Order

You must split **chronologically**:

- **Train on earlier dates** (past)
- **Test on later dates** (future)

```python
# Sort by date first
df = df.sort_values('Date')

# Calculate split point
train_size = int(len(df) * 0.8)

# Split chronologically
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_test = test_df.drop('target', axis=1)
y_test = test_df['target']
```

### Using TimeSeriesSplit

scikit-learn provides `TimeSeriesSplit` for cross-validation:

```python
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for train_idx, test_idx in tscv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # Train and evaluate
```

Each fold respects temporal order: training is always before testing.

### When to Use Time-Based Splitting

- Stock price prediction
- Sales forecasting
- Weather prediction
- Sensor data analysis
- Any data with temporal dependency

**Always respect temporal order when modeling time-dependent data.**

In [ ]:
# Demonstrate time-based splitting

from sklearn.model_selection import TimeSeriesSplit

# Create time-series data
dates = pd.date_range('2020-01-01', periods=100, freq='D')
df_ts = pd.DataFrame({
    'date': dates,
    'value': np.cumsum(np.random.randn(100)),  # Random walk (time-dependent)
    'target': np.random.randint(0, 2, 100)
})

print("=" * 60)
print("TIME-BASED SPLITTING")
print("=" * 60)
print()

# Correct: Chronological split
print("CORRECT: Chronological split")
print("-" * 60)
train_size = int(len(df_ts) * 0.8)
train_ts = df_ts.iloc[:train_size]
test_ts = df_ts.iloc[train_size:]

print(f"Training dates: {train_ts['date'].min().date()} to {train_ts['date'].max().date()}")
print(f"Testing dates: {test_ts['date'].min().date()} to {test_ts['date'].max().date()}")
print("✓ Training is before testing (respects time order)")
print()

# Demonstrate TimeSeriesSplit for cross-validation
print("TimeSeriesSplit (for cross-validation):")
print("-" * 60)
tscv = TimeSeriesSplit(n_splits=3)

fold = 1
for train_idx, test_idx in tscv.split(df_ts):
    train_fold = df_ts.iloc[train_idx]
    test_fold = df_ts.iloc[test_idx]
    
    print(f"Fold {fold}:")
    print(f"  Train: {train_fold['date'].min().date()} to {train_fold['date'].max().date()}")
    print(f"  Test: {test_fold['date'].min().date()} to {test_fold['date'].max().date()}")
    fold += 1

## Understanding Data Leakage During Splitting

### Types of Leakage

#### Leakage 1: Fitting Before Splitting

```python
# WRONG
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Fitted on all data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**Fix:** Split first, then fit on training data only.

#### Leakage 2: Feature Selection on Full Dataset

```python
# WRONG
from sklearn.feature_selection import SelectKBest

selector = SelectKBest(k=10)
X_selected = selector.fit_transform(X, y)  # Fitted on all data and all y
X_train, X_test, y_train, y_test = train_test_split(X_selected, y)
```

**Fix:** Split first, then perform feature selection on training data.

#### Leakage 3: Resampling (SMOTE) Before Splitting

```python
# WRONG
from imblearn.over_sampling import SMOTE

smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)  # Applied to all data
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled)
```

**Why it leaks:**
- Test set synthetic samples are created based on all data
- Test samples may be similar to training samples
- Model sees "new" samples that are actually synthetic versions of data it trained on

**Fix:**
```python
# RIGHT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE()
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)  # Only training
# Test set remains unchanged
```

#### Leakage 4: Using Target-Based Statistics

```python
# WRONG
# Computing class weights from full dataset
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
X_train, X_test, y_train, y_test = train_test_split(X, y)
# Train model with class_weights computed from all data
```

**Fix:** Compute weights from training data only.

### Summary: Prevention Strategy

**Any operation that uses information from the entire dataset must be done AFTER splitting.**

Safe sequence:
1. Split
2. Fit transformations on training data
3. Apply transformations to test data
4. Train model
5. Evaluate on test set

## Verifying the Split

After splitting, verification prevents silent errors. Check:

### 1. Shapes

```python
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
```

Expected: training size ≈ 80% of total, test size ≈ 20%

### 2. Class Distribution (Classification)

```python
print("Train class distribution:")
print(y_train.value_counts(normalize=True))

print("Test class distribution:")
print(y_test.value_counts(normalize=True))
```

Expected: similar proportions if stratified

### 3. No Overlap

```python
# Check no samples appear in both sets
train_indices = set(X_train.index)
test_indices = set(X_test.index)
overlap = train_indices.intersection(test_indices)

print(f"Overlap: {len(overlap)} samples")  # Should be 0
```

### 4. Target Value Ranges (Regression)

```python
print("Training y range:", y_train.min(), "to", y_train.max())
print("Testing y range:", y_test.min(), "to", y_test.max())
```

Expected: similar ranges (not all values in one set)

In [ ]:
# Verification checklist

print("=" * 60)
print("SPLIT VERIFICATION CHECKLIST")
print("=" * 60)
print()

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Check 1: Shapes
print("✓ SHAPES")
print(f"  Full dataset: {X.shape}")
print(f"  Training: {X_train_final.shape} ({len(X_train_final)/len(X)*100:.1f}%)")
print(f"  Testing: {X_test_final.shape} ({len(X_test_final)/len(X)*100:.1f}%)")
print()

# Check 2: Class distribution
print("✓ CLASS DISTRIBUTION")
print(f"  Full set: {y.value_counts(normalize=True).to_dict()}")
print(f"  Training: {y_train_final.value_counts(normalize=True).to_dict()}")
print(f"  Testing: {y_test_final.value_counts(normalize=True).to_dict()}")
print()

# Check 3: No overlap
print("✓ NO OVERLAP CHECK")
train_idx = set(X_train_final.index)
test_idx = set(X_test_final.index)
overlap = len(train_idx.intersection(test_idx))
print(f"  Samples in both sets: {overlap} (should be 0)")
print()

# Check 4: Target value ranges
print("✓ TARGET VALUE RANGES")
print(f"  Full target: {y.min()} to {y.max()}")
print(f"  Training target: {y_train_final.min()} to {y_train_final.max()}")
print(f"  Testing target: {y_test_final.min()} to {y_test_final.max()}")
print()

print("All checks passed! Split is valid.")

## Understanding Cross-Validation

### The Limitation of Single Train-Test Split

A single train-test split produces **one performance estimate**.

- If you're unlucky, the test set may be easier or harder than average
- Performance estimate may be unstable
- Small datasets: one split wastes data

### Cross-Validation Solution

Cross-validation splits training data into **multiple folds**:

- Train model multiple times, each time using different fold as validation
- Produces multiple performance estimates
- More stable and reliable performance estimate

**Example: 5-Fold Cross-Validation**

```python
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

scores = cross_val_score(
    model, X_train, y_train,
    cv=5,  # 5 folds
    scoring='accuracy'
)

print(f"CV scores: {scores}")
print(f"Mean: {scores.mean():.3f}")
print(f"Std: {scores.std():.3f}")
```

### Critical Point: CV ≠ Test Set Replacement

**Cross-validation does NOT replace the test set.**

| Phase | Purpose | Data |
|-------|---------|------|
| Development | Tune hyperparameters, model selection | Training set + Cross-validation |
| Final evaluation | Measure generalization | Held-out test set |

**Workflow:**
1. Split into training and test sets
2. Use cross-validation on training set for hyperparameter tuning
3. Train final model on full training set with best hyperparameters
4. Evaluate ONCE on test set (final evaluation)

### When to Use Cross-Validation

- Small datasets (< 1000 samples) → more stable estimates
- Hyperparameter tuning → get reliable scores
- Model selection → compare multiple algorithms
- Confidence intervals → understand variability

### Common Cross-Validation Strategies

- **k-Fold** (standard) → simple, works well
- **Stratified k-Fold** (classification) → preserves class balance
- **Time Series Split** (temporal) → respects time order
- **Leave-One-Out** (very small data) → expensive, rarely used

## Best Practices Summary

### Before Splitting

- [ ] Ensure data is loaded and inspected
- [ ] Verify no extreme leakage issues (like duplicates)
- [ ] Decide: random or stratified split?
- [ ] Decide: chronological split for time-series?
- [ ] Choose fixed random_state for reproducibility

### During Splitting

- [ ] **Split BEFORE any preprocessing fitting**
- [ ] Use stratified split for classification
- [ ] Use time-based split for temporal data
- [ ] Always set random_state

### After Splitting

- [ ] Verify shapes and proportions
- [ ] Check class balance (classification)
- [ ] Fit preprocessing ONLY on training data
- [ ] Apply to test data with transform() only
- [ ] Protect test set — never use for hyperparameter tuning
- [ ] Final evaluation: always on test set

### Throughout Development

- [ ] Use cross-validation on training data for tuning
- [ ] Never touch test set until the very end
- [ ] Document split strategy
- [ ] Report final performance on test set only

## Documenting the Split Strategy

Professional ML projects require clear documentation of splitting decisions. This ensures reproducibility and explains reasoning to collaborators.

### Example Documentation

```markdown
## Data Splitting Strategy

### Overview
- **Total samples:** 5,000
- **Train-test ratio:** 80-20 (4,000 training, 1,000 testing)
- **Splitting method:** Stratified random split
- **Random state:** 42 (for reproducibility)

### Rationale
- Stratified split chosen to preserve class balance (binary classification)
- 80-20 split is standard for this dataset size
- Fixed random_state ensures reproducibility across runs

### Class Distribution
- Full dataset: 70% class 0, 30% class 1
- Training set: 70% class 0, 30% class 1 (preserved)
- Testing set: 70% class 0, 30% class 1 (preserved)

### Preprocessing After Split
- StandardScaler fitted on training data only
- Scaling parameters applied to test set (no refitting)
- No data leakage

### Cross-Validation
- 5-fold stratified cross-validation used for hyperparameter tuning
- Test set remained untouched during development
- Final evaluation performed only on held-out test set

### Implementation
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```
```

### Why Documentation Matters

- **Reproducibility** — others can recreate exact same split
- **Transparency** — explains design decisions
- **Audit trail** — justifies evaluation methodology
- **Problem diagnosis** — if performance differs, split strategy is documented for investigation

## Common Mistakes in Data Splitting

### Mistake 1: Evaluating on Training Data

```python
# WRONG
model.fit(X_train, y_train)
accuracy = model.score(X_train, y_train)  # Evaluating on training data!
print(f"Model accuracy: {accuracy}")  # Artificially high
```

**Why it fails:** Training accuracy is always optimistic. It measures memorization, not generalization.

**Fix:** Always evaluate on test set.

```python
# RIGHT
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)  # Test set evaluation
```

### Mistake 2: Scaling Before Splitting

```python
# WRONG
X_scaled = StandardScaler().fit_transform(X)  # Fitted on all data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**Why it fails:** Test data statistics leak into training through scaler parameters.

**Fix:** Split first, fit on training only.

### Mistake 3: Forgetting Stratification in Classification

```python
# WRONG - For imbalanced classification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2  # No stratification
)
```

**Why it fails:** Class imbalance may differ between train and test, causing unstable evaluation.

**Fix:** Add stratify parameter.

```python
# RIGHT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)
```

### Mistake 4: Random Shuffling for Time-Series Data

```python
# WRONG
X_train, X_test, y_train, y_test = train_test_split(
    X_ts, y_ts, test_size=0.2, shuffle=True  # Random shuffling!
)
```

**Why it fails:** Breaks temporal dependency. Model trains on future data.

**Fix:** Use chronological split.

```python
# RIGHT
train_idx = int(len(X_ts) * 0.8)
X_train, X_test = X_ts.iloc[:train_idx], X_ts.iloc[train_idx:]
y_train, y_test = y_ts.iloc[:train_idx], y_ts.iloc[train_idx:]
```

### Mistake 5: Using Test Set During Hyperparameter Tuning

```python
# WRONG
best_accuracy = 0
for C in [0.1, 1, 10]:
    model = SVC(C=C)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)  # Tuning on test set!
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_C = C
```

**Why it fails:** You're optimizing hyperparameters on test data. Test set is no longer unseen.

**Fix:** Use cross-validation on training data.

```python
# RIGHT
from sklearn.model_selection import cross_val_score

best_score = 0
for C in [0.1, 1, 10]:
    model = SVC(C=C)
    scores = cross_val_score(model, X_train, y_train, cv=5)  # CV on training only
    if scores.mean() > best_score:
        best_score = scores.mean()
        best_C = C
```

**Each of these mistakes leads to invalid performance estimates and unreliable models.**

## Closing Reflection

### The Core Principle

**A powerful model trained on improperly split data is not a powerful model — it is a misleading one.**

The train-test boundary is sacred. Once contaminated, evaluation becomes meaningless.

### What Proper Splitting Protects

- **Against overfitting** — test set catches memorization
- **Against data leakage** — preprocessing doesn't leak information
- **Against bias** — stratification preserves class balance
- **Against false confidence** — shows real generalization ability

### The Workflow You Should Follow

1. **Inspect data** (previous lesson)
2. **Split intentionally** (this lesson)
3. **Preprocess carefully** (fit on training only)
4. **Develop thoughtfully** (use cross-validation on training)
5. **Evaluate honestly** (test set, once, at the end)

### Remember

- **Split carefully.** Decide: random or stratified? Time-series or standard?
- **Validate the split.** Verify shapes, distributions, no leakage.
- **Protect the test set.** Never touch it until final evaluation.
- **Document decisions.** Explain your splitting strategy.

**Honest evaluation is the foundation of trustworthy machine learning.**

## Bonus Content 🎁

This section is optional, and learners who want to explore the topics covered so far can utilize the materials provided below.

### Official Documentation

- **scikit-learn `train_test_split`:** https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- **scikit-learn Cross-Validation:** https://scikit-learn.org/stable/modules/cross_validation.html
- **TimeSeriesSplit Documentation:** https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html

### Data Leakage Resources

- **Kaggle: "Data Leakage"** — comprehensive guide on leakage types and prevention
- **"Leakage and the Reproducibility Crisis" (paper)** — research on data leakage in ML
- **Coursera ML course** — Andrew Ng discusses train-test splits in depth

### Advanced Topics

**Stratified k-Fold for Imbalanced Classification:**
```python
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, test_idx in skf.split(X, y):
    # Each fold preserves class balance
```

**GroupKFold (for grouped data):**
```python
from sklearn.model_selection import GroupKFold

# Useful when samples belong to groups (e.g., patient ID)
# Prevents data leakage across groups
gkf = GroupKFold(n_splits=5)
for train_idx, test_idx in gkf.split(X, y, groups=patient_ids):
    # Groups are never split across train and test
```

**StratifiedShuffleSplit (for repeated splits):**
```python
from sklearn.model_selection import StratifiedShuffleSplit

# Useful when you want multiple independent train-test splits
sss = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X, y):
    # Get 10 different stratified splits
```

### Preventing Leakage: Pipelines

scikit-learn `Pipeline` encapsulates preprocessing and model, ensuring no leakage:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Pipeline ensures preprocessing is fit only on training data
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)  # Preprocessing applied correctly
```